# CNN-DAE — Avaliação Estatística Final

**Arquivos necessários (célula 4):** `best_model.keras`, `data_io.py`, `contaminate.py`, `dataset.py`, `model.py`, `metrics.py`, `comparacao_convencionais.csv`

## 1 · Setup

In [ ]:
!pip install -q wfdb scipy
import tensorflow as tf, numpy as np, pandas as pd
from scipy import stats
print('TF:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 2 · Upload dos arquivos

Envie os **7 arquivos** abaixo:

| Arquivo | Origem |
|---|---|
| `best_model.keras` | `cnn-dae/checkpoints/` |
| `data_io.py` | `cnn-dae/src/` |
| `contaminate.py` | `cnn-dae/src/noise/` |
| `dataset.py` | `cnn-dae/src/models/` |
| `model.py` | `cnn-dae/src/models/` |
| `metrics.py` | `src/metrics/` |
| `comparacao_convencionais.csv` | `figures/` |

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Recebidos:', list(uploaded.keys()))

## 3 · Estrutura de pacotes

In [ ]:
import shutil, pathlib, sys

for folder in ['src', 'src/noise', 'src/models', 'src/metrics']:
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)
    pathlib.Path(f'{folder}/__init__.py').touch()

for src, dst in [
    ('data_io.py',    'src/data_io.py'),
    ('contaminate.py','src/noise/contaminate.py'),
    ('dataset.py',    'src/models/dataset.py'),
    ('model.py',      'src/models/model.py'),
    ('metrics.py',    'src/metrics/metrics.py'),
]:
    if pathlib.Path(src).exists():
        shutil.copy(src, dst)

sys.path.insert(0, '/content')
print('OK')

## 4 · Carregar modelo e dados

In [ ]:
from src.data_io import ensure_datasets
from src.models.dataset import build_data_pools, build_test_table

model = tf.keras.models.load_model('best_model.keras')
print(f'Parâmetros: {model.count_params():,}')

ensure_datasets()
pools      = build_data_pools()
test_table = build_test_table(pools)
print(f'Janelas de teste: {pools.ecg_test_windows.shape[0]}')
print(f'Combinações (ruído × SNR): {len(test_table)}')

## 5 · Módulo A — Métricas por janela

Gera um DataFrame com uma linha por `(registro_idx, ruído, snr_in)`. Usa todas as métricas de `metrics.py` + MAE/MSE.

In [ ]:
from src.metrics.metrics import snr_db, snr_improvement, rmse, prd, correlation

rows = []
for (noise_type, snr_in), (X, Y) in test_table.items():
    Y_hat = model.predict(X, verbose=0)  # (N, 1024, 1)
    for i in range(len(X)):
        c = Y[i, :, 0]
        n = X[i, :, 0]
        f = Y_hat[i, :, 0]
        rows.append({
            'noise'   : noise_type,
            'snr_in'  : snr_in,
            'window'  : i,
            'mae'     : float(np.mean(np.abs(f - c))),
            'mse'     : float(np.mean((f - c) ** 2)),
            'snr_out' : snr_db(c, f),
            'delta_snr': snr_improvement(c, n, f),
            'rmse'    : rmse(c, f),
            'prd'     : prd(c, f),
            'corr'    : correlation(c, f),
        })

df = pd.DataFrame(rows)
print(f'{len(df)} linhas  |  colunas: {list(df.columns)}')
df.head()

## 6 · Módulo B — Estatística descritiva

In [ ]:
METRICS = ['mae', 'mse', 'snr_out', 'delta_snr', 'rmse', 'prd', 'corr']

def describe_group(groupby_cols):
    agg = {m: ['mean', 'median', 'std',
               lambda x: x.quantile(0.25), lambda x: x.quantile(0.75),
               lambda x: x.quantile(0.05), lambda x: x.quantile(0.95)]
           for m in METRICS}
    result = df.groupby(groupby_cols).agg(agg)
    result.columns = ['_'.join([m, s if not callable(s) else
                                ('Q25' if i % 7 == 3 else 'Q75' if i % 7 == 4 else
                                 'P05' if i % 7 == 5 else 'P95')])
                      for i, (m, s) in enumerate(
                          [(m, s) for m in METRICS
                           for s in ['mean','median','std','Q25','Q75','P05','P95']])]
    return result

# por ruído
print('=== Por tipo de ruído ===')
sum_noise = df.groupby('noise')[METRICS].agg(['mean','median','std']).round(4)
print(sum_noise)

# por SNR de entrada
print('\n=== Por SNR de entrada ===')
sum_snr = df.groupby('snr_in')[METRICS].agg(['mean','median','std']).round(4)
print(sum_snr)

# % janelas onde o modelo piorou (delta_snr < 0)
print('\n=== % janelas com ΔSNR < 0 (modelo piorou) ===')
fail_rate = df.groupby(['noise','snr_in']).apply(
    lambda g: 100 * (g['delta_snr'] < 0).mean()
).rename('fail_pct').reset_index()
print(fail_rate.pivot(index='noise', columns='snr_in', values='fail_pct').round(1))

## 7 · Módulo C — Comparação com filtros convencionais (Wilcoxon)

Para cada par (ruído, SNR) disponível no CSV, testa H₀: modelo == baseline via Wilcoxon signed-rank. Retorna estatística W, p-value e tamanho de efeito r = Z/√N.

In [ ]:
baseline = pd.read_csv('comparacao_convencionais.csv')
print(baseline)

wilcoxon_rows = []
for _, brow in baseline.iterrows():
    noise, snr_in = brow['ruido'], int(brow['snr_in_alvo'])
    sub = df[(df['noise'] == noise) & (df['snr_in'] == snr_in)]
    if sub.empty:
        continue
    for metric, b_val in [('snr_out', brow['snr_out_db']),
                          ('prd',     brow['prd_pct']),
                          ('corr',    brow['corr'])]:
        model_vals = sub[metric].values
        # Wilcoxon exige variância; se todos iguais ao baseline, pula
        diffs = model_vals - b_val
        if np.all(diffs == 0):
            continue
        stat, pval = stats.wilcoxon(diffs, alternative='two-sided')
        n = len(model_vals)
        z = stats.norm.ppf(pval / 2)  # z aproximado
        r = abs(z) / np.sqrt(n)       # tamanho de efeito
        wilcoxon_rows.append({
            'filtro'  : brow['filtro'],
            'noise'   : noise,
            'snr_in'  : snr_in,
            'metric'  : metric,
            'model_mean': model_vals.mean(),
            'baseline': b_val,
            'W'       : stat,
            'p_value' : pval,
            'effect_r': r,
            'sig'     : '*' if pval < 0.05 else '',
        })

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
pd.set_option('display.max_rows', 60)
print(wilcoxon_df.to_string(index=False))

## 8 · Módulo D — Intervalos de confiança (Bootstrap)

B = 1000 reamostras das janelas por célula. IC 95% para PRD médio, correlação média e ΔSNR médio.

In [ ]:
B    = 1000
RNG  = np.random.default_rng(42)
BOOT_METRICS = ['prd', 'corr', 'delta_snr']

boot_rows = []
for (noise, snr_in), grp in df.groupby(['noise', 'snr_in']):
    vals = grp[BOOT_METRICS].values  # (N, 3)
    boots = np.array([
        vals[RNG.integers(0, len(vals), len(vals))].mean(axis=0)
        for _ in range(B)
    ])  # (B, 3)
    for j, m in enumerate(BOOT_METRICS):
        lo, hi = np.percentile(boots[:, j], [2.5, 97.5])
        boot_rows.append({
            'noise'   : noise,
            'snr_in'  : snr_in,
            'metric'  : m,
            'estimate': vals[:, j].mean(),
            'ci_low'  : lo,
            'ci_high' : hi,
        })

boot_df = pd.DataFrame(boot_rows)
print(boot_df.pivot_table(
    index=['noise','snr_in'], columns='metric',
    values=['estimate','ci_low','ci_high']
).round(4))

## 9 · Módulo E — Análise de falha

Janelas onde ΔSNR < 0 por condição + visualização das N piores.

In [ ]:
import matplotlib.pyplot as plt

fails = df[df['delta_snr'] < 0].copy()
print(f'Total de falhas: {len(fails)} / {len(df)} ({100*len(fails)/len(df):.1f}%)')
print('\nFalhas por (ruído, SNR):')
print(fails.groupby(['noise','snr_in']).size().rename('n_fails'))

# Piores N janelas globais
N_WORST = 6
worst = df.nsmallest(N_WORST, 'delta_snr')[['noise','snr_in','window','delta_snr','prd','corr']]
print('\nPiores janelas:')
print(worst.to_string(index=False))

In [ ]:
# Visualização das piores janelas
N_VIS = min(4, len(worst))
t = np.arange(1024) / 360

fig, axes = plt.subplots(N_VIS, 3, figsize=(15, 3.2 * N_VIS), sharex=True)
if N_VIS == 1:
    axes = axes[np.newaxis, :]

for row_i, (_, wrow) in enumerate(worst.head(N_VIS).iterrows()):
    noise, snr_in, win_i = wrow['noise'], int(wrow['snr_in']), int(wrow['window'])
    X_cell, Y_cell = test_table[(noise, snr_in)]
    Y_hat_cell     = model.predict(X_cell[win_i:win_i+1], verbose=0)
    x_, y_, yhat_  = X_cell[win_i,:,0], Y_cell[win_i,:,0], Y_hat_cell[0,:,0]

    axes[row_i, 0].plot(t, x_,    color='tab:orange', lw=0.7)
    axes[row_i, 0].set_title(f'{noise} | SNR={snr_in}dB | janela={win_i} — Ruidoso')
    axes[row_i, 1].plot(t, yhat_, color='tab:blue',   lw=0.7)
    axes[row_i, 1].set_title(f'Filtrado  ΔSNR={wrow["delta_snr"]:.2f}dB')
    axes[row_i, 2].plot(t, y_,    color='tab:green',  lw=0.7)
    axes[row_i, 2].set_title('Referência')
    for ax in axes[row_i]: ax.grid(True, alpha=0.3)

axes[-1,1].set_xlabel('Tempo (s)')
plt.suptitle('Piores janelas (menor ΔSNR)', fontsize=12)
plt.tight_layout()
plt.show()

## 10 · Boxplots por ruído e por SNR

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['delta_snr', 'prd', 'corr']):
    groups = [df[df['noise'] == n][metric].values
              for n in df['noise'].unique()]
    ax.boxplot(groups, labels=df['noise'].unique(), patch_artist=True)
    ax.axhline(0 if metric == 'delta_snr' else
               (9 if metric == 'prd' else 0.95),
               color='red', lw=1, linestyle='--', alpha=0.7)
    ax.set_title(metric)
    ax.set_xlabel('Tipo de ruído')
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribuição por tipo de ruído  [linha vermelha = limiar referência]')
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes2, ['delta_snr', 'prd', 'corr']):
    groups = [df[df['snr_in'] == s][metric].values
              for s in sorted(df['snr_in'].unique())]
    ax.boxplot(groups, labels=sorted(df['snr_in'].unique()), patch_artist=True)
    ax.set_title(metric)
    ax.set_xlabel('SNR de entrada (dB)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribuição por SNR de entrada')
plt.tight_layout()
plt.show()

## 11 · Exportar resultados

In [ ]:
df.to_csv('resultados_por_janela.csv', index=False)
wilcoxon_df.to_csv('wilcoxon_vs_baseline.csv', index=False)
boot_df.to_csv('bootstrap_ic95.csv', index=False)
print('CSVs salvos: resultados_por_janela.csv | wilcoxon_vs_baseline.csv | bootstrap_ic95.csv')